# Section 10 — Extrinsic Retrieval Evaluation

This notebook evaluates whether the generated temporal hypergraph hierarchy can
retrieve relevant concepts for benchmark questions.

Unlike Section 09, which measures intrinsic hierarchy properties, this section
uses external benchmark supervision:

- questions.csv
- ground_truth.json

The hierarchy is treated as a retrieval structure.

The evaluation measures whether relevant supernodes are retrieved before answer
generation.

## Clone repository

In [1]:
from pathlib import Path

REPO_DIR = Path(
    "/content/tkh-hierarchy-project"
)


if not REPO_DIR.exists():

    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git


%cd /content/tkh-hierarchy-project

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 147 (delta 89), reused 93 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 17.09 MiB | 29.07 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/tkh-hierarchy-project


In [2]:
!ls

artifacts  LICENSE    README.md		      t1_statistics.json
data	   notebooks  snapshot_metadata.json


## Imports

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

from collections import defaultdict

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Define paths

In [ ]:
PROJECT_DIR = Path(
    "/content/tkh-hierarchy-project"
)


DATA_DIR = (
    PROJECT_DIR
    /
    "data"
)


LABELLED_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "hierarchy"
    /
    "labelled"
)


TKH_PATH = (
    DATA_DIR
    /
    "tkh_collection10.json"
)


QUESTIONS_PATH = (
    DATA_DIR
    /
    "questions.csv"
)


GROUND_TRUTH_PATH = (
    DATA_DIR
    /
    "ground_truth.json"
)


print(TKH_PATH)
print(QUESTIONS_PATH)
print(GROUND_TRUTH_PATH)
print(LABELLED_DIR)

## Load TKH graph

In [ ]:
with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:

    tkh = json.load(f)


nodes = tkh["nodes"]

hyperedges = tkh["hyperedges"]


node_lookup = {

    node["id"]:
        node

    for node in nodes

}


print(
    "Nodes:",
    len(nodes)
)


print(
    "Hyperedges:",
    len(hyperedges)
)

## Load labelled hierarchies

files are:
```text
hierarchy_2020_labelled.json
hierarchy_2022_labelled.json
hierarchy_2024_labelled.json
hierarchy_2026_labelled.json

In [ ]:
SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026
]


labelled_hierarchies = {}


for year in SNAPSHOT_YEARS:

    path = (
        LABELLED_DIR
        /
        f"hierarchy_{year}_labelled.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        labelled_hierarchies[year] = json.load(f)



print(
    labelled_hierarchies.keys()
)

In [ ]:
questions_df = pd.read_csv(
    QUESTIONS_PATH,
    sep=";"
)


print(
    questions_df.shape
)


questions_df.head()

## Load ground truth

In [ ]:
with open(
    GROUND_TRUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    ground_truth = json.load(f)


print(
    "Questions in ground truth:",
    len(ground_truth)
)

## Verify benchmark structure

In [ ]:
for qid, item in list(
    ground_truth.items()
)[:3]:

    print(
        qid,
        item.keys()
    )

## Build retrieval candidates

Important change:

We include:

* method
* component
* dataset
* technique

because your benchmark contains all of them.

In [ ]:
ALLOWED_TYPES = {

    "method",
    "component",
    "dataset",
    "technique"

}


retrieval_units = []


for node_id, node in node_lookup.items():


    if node.get("type") not in ALLOWED_TYPES:
        continue


    label = node.get(
        "surface_form"
    )


    if label is None:
        continue


    retrieval_units.append({

        "persistent_id":
            node_id,

        "label":
            label,

        "type":
            node["type"],

        "text":
            (
                node["type"]
                +
                ": "
                +
                label
            )

    })


retrieval_df = pd.DataFrame(
    retrieval_units
)


print(
    "Candidates:",
    len(retrieval_df)
)


print(
    retrieval_df["type"].value_counts()
)

## Load embedding model

In [ ]:
model = SentenceTransformer(
    "all-mpnet-base-v2"
)

## Encode TKH entities

In [ ]:
retrieval_embeddings = model.encode(

    retrieval_df["text"].tolist(),

    normalize_embeddings=True

)


print(
    retrieval_embeddings.shape
)

## Encode questions

In [ ]:
question_texts = [

    "Find scientific methods and related concepts: "
    +
    str(q)

    for q in questions_df["question"]

]


question_embeddings = model.encode(

    question_texts,

    normalize_embeddings=True

)


print(
    question_embeddings.shape
)

## Retrieval function

In [ ]:
def retrieve_top_k(
    embedding,
    k=5
):

    scores = cosine_similarity(

        embedding.reshape(1,-1),

        retrieval_embeddings

    )[0]


    indices = np.argsort(
        scores
    )[::-1][:k]


    results = []


    for idx in indices:

        row = retrieval_df.iloc[idx]


        results.append({

            "persistent_id":
                row["persistent_id"],

            "label":
                row["label"],

            "type":
                row["type"],

            "score":
                float(
                    scores[idx]
                )

        })


    return results

## Retrieve

In [ ]:
retrieval_results = []


for idx, embedding in enumerate(
    question_embeddings
):


    retrieval_results.append({

        "question_id":
            questions_df.iloc[idx]["question_id"],

        "retrieved":
            retrieve_top_k(
                embedding,
                k=5
            )

    })


print(
    len(retrieval_results)
)

## Debug retrieval BEFORE evaluation

In [ ]:
for result in retrieval_results[:]:

    print("="*80)

    print(
        result["question_id"]
    )


    for r in result["retrieved"]:

        print(
            r["label"],
            "|",
            r["type"],
            "|",
            round(
                r["score"],
                4
            )
        )

## Canonical mapping

In [ ]:
method_aliases = {


    # MACE
    "MACE-MP-0":
        "MACE",


    # GAP
    "Gaussian Approximation Potential (GAP) for silicon":
        "GAP",


    # CHGNet
    "CHGNet":
        "CHGNet",


    # NequIP
    "NequIP":
        "NequIP",


    # ACE
    "ACE with message passing":
        "ACE",

    "CACE":
        "ACE",


    # Deep potential family
    "Deep Potential":
        "DeePMD",

    "Deep Potential MD":
        "DeePMD",

    "Deep Potential Molecular Dynamics":
        "DeePMD",

    "Deep Potential method":
        "DeePMD",


    "DeePMD-kit v2":
        "DeePMD",


    # DeepH
    "DeepH-E3":
        "DeepH",


    # Orb
    "Orb-v2":
        "Orb",

    "Orb-v3":
        "Orb",


    # SevenNet
    "SevenNet ML-IAP":
        "SevenNet",

    "SevenNet-0":
        "SevenNet",


    # HamGNN
    "Universal HamGNN Hamiltonian model":
        "HamGNN",


    # MPtrj
    "MPtrj dataset":
        "MPtrj",


    # UMA
    "UMA-S":
        "UMA",

    "UMA-M":
        "UMA",

    "UMA-L":
        "UMA"

}

## Canonicalization function

In [ ]:
def canonicalize(label):

    label = str(label).strip()


    return method_aliases.get(
        label,
        label
    )

## Gold extraction

In [ ]:
def extract_gold_methods(item):

    return set(
        item.get(
            "expected_methods",
            []
        )
    )

## Evaluation

In [ ]:
evaluation_rows = []


for result in retrieval_results:


    qid = str(
        result["question_id"]
    )


    gold = {

        canonicalize(x)

        for x in extract_gold_methods(
            ground_truth[qid]
        )

    }


    predicted = {

        canonicalize(
            r["label"]
        )

        for r in result["retrieved"]

    }


    hits = (
        gold &
        predicted
    )


    precision = (

        len(hits)

        /

        max(
            len(predicted),
            1
        )

    )


    recall = (

        len(hits)

        /

        max(
            len(gold),
            1
        )

    )


    f1 = (

        2 *
        precision *
        recall

        /

        max(
            precision + recall,
            1e-9
        )

    )


    evaluation_rows.append({

        "question_id":
            qid,

        "gold_methods":
            list(gold),

        "predicted_methods":
            list(predicted),

        "hits":
            list(hits),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1

    })


retrieval_eval_df = pd.DataFrame(
    evaluation_rows
)


retrieval_eval_df

## Final metrics

In [ ]:
summary = {

    "questions":
        len(retrieval_eval_df),


    "mean_precision":
        retrieval_eval_df["precision"].mean(),


    "mean_recall":
        retrieval_eval_df["recall"].mean(),


    "mean_f1":
        retrieval_eval_df["f1"].mean()

}


summary

## Save result

In [ ]:
retrieval_eval_df.to_csv(
    "retrieval_evaluation_results.csv",
    index=False
)


print(
    "Saved."
)

# Section 10 has established


*   The constructed hierarchy was evaluated through an external downstream retrieval task.
*   Benchmark questions and expected methods were used to assess whether the generated TKH representations can support information discovery.
*   Retrieval was performed using semantic similarity between question representations and TKH entities.
*   Retrieved entities were compared against benchmark expected methods after applying entity normalization and alias handling.
*   Evaluation metrics were calculated using:

    - precision;
    - recall;
    - F1 score.

*   The evaluation measures whether the hierarchical TKH representation can recover relevant scientific concepts from external queries.
*   The results demonstrate both the strengths and limitations of semantic retrieval over the constructed abstraction.
*   Retrieval errors provide insight into remaining challenges, including vocabulary mismatch between benchmark terminology and TKH surface forms.


---

## Contribution to the project

The complete pipeline has now progressed from raw temporal hypergraph data to an
externally evaluated knowledge abstraction system:
```text
Raw TKH
|
↓
Validated hypergraph structure
|
↓
Temporal snapshots
|
↓
Descriptive analysis
|
↓
Semantic representation
|
↓
Hypergraph-native abstraction objective
|
↓
Multi-resolution hierarchy
|
↓
Temporal coupling
|
↓
Semantic labelling
|
↓
Intrinsic evaluation
|
↓
Extrinsic retrieval evaluation
```

## Git Push

In [ ]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/10_extrinsic_retrieval.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/10_extrinsic_retrieval.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())

In [ ]:
import shutil

shutil.copy2(CLEAN, REPO_FILE)

print("Clean notebook copied into repository.")

In [ ]:
!git status

In [ ]:
!git add -A

In [ ]:
!git status

In [ ]:
!git config --global user.name "mohamadghoroobi"
!git config --global user.email "m.ghoroobi@gmail.com"

In [ ]:
commit_message = """ feat(evaluation): implement extrinsic retrieval evaluation over TKH hierarchy

Evaluate the practical usefulness of the constructed temporal multi-resolution abstraction through downstream retrieval.


- load benchmark questions and ground truth annotations

- construct retrieval candidates from TKH entities

- include heterogeneous entity types:
  method, component, dataset, and technique

- generate semantic embeddings for retrieval

- perform top-k similarity-based retrieval

- introduce entity normalization and alias handling for benchmark comparison

- compare retrieved concepts against expected methods

- compute precision, recall, and F1 metrics

- document vocabulary mismatch limitations between benchmark terminology and TKH surface forms

- provide final evaluation summary for the complete abstraction pipeline

"""

with open("/tmp/commit_message.txt", "w", encoding="utf-8") as f:
    f.write(commit_message)

In [ ]:
!git commit -F /tmp/commit_message.txt

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

assert token, "GITHUB_TOKEN not found"
print("Token loaded successfully")

In [ ]:
import os
import subprocess
from pathlib import Path

username = "mohamadghoroobi"

env = os.environ.copy()

env["GITHUB_USER"] = "mohamadghoroobi"
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"


askpass = Path("/tmp/git_askpass.sh")

askpass.write_text(
"""#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USER" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)

askpass.chmod(0o700)

env["GIT_ASKPASS"] = str(askpass)

print("Git authentication prepared")

In [ ]:
subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    check=True
)

print("Push completed successfully")

In [ ]:
!git status